# Chapter 4: 微分可能最適化

**SLAM Handbook Chapter 4** — 最適化ソルバーを「通して」勾配を逆伝播する技術。

## このNotebookで学ぶこと

1. **Bilevel最適化** — NN(フロントエンド) + NLS(バックエンド) のend-to-end学習
2. **Unrolled Differentiation** — 反復を展開してAutoDiffで微分
3. **Implicit Differentiation** — 最適性条件から間接的に勾配を計算 (IFT)
4. **マニフォールド上の微分** — Lie群のヤコビアン (right/left)
5. **デモ: 微分可能な最小二乗** — パラメータをbackpropで学習

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 4.1 Bilevel最適化の動機

**SLAM Handbook Section 4.2, Fig. 4.1**: 現代のSLAMはフロントエンド（NN）とバックエンド（最適化）に分かれている。

$$\mathbf{y}^* = \arg\min_{\mathbf{y}} \mathcal{U}(\mathbf{y}, \mathbf{x}^*) \quad \text{(Upper-Level: NNの学習)}$$
$$\text{s.t.} \quad \mathbf{x}^* = \arg\min_{\mathbf{x}} \mathcal{L}(\mathbf{y}, \mathbf{x}) \quad \text{(Lower-Level: NLS最適化)}$$

**課題**: $\frac{\partial \mathbf{x}^*}{\partial \mathbf{y}}$ をどう計算するか？ → 最適化ソルバーを「微分可能」にする必要がある。

### 2つのアプローチ
- **Unrolled**: 反復を展開してautodiff（メモリ大、近似的）
- **Implicit**: 最適性条件 $\nabla_\mathbf{x}\mathcal{L}=0$ の陰関数定理で計算（正確、Hessian必要）

In [ ]:
# =============================================================
# Toy例題: パラメータyを持つ最小二乗問題
# =============================================================
# Lower-level: x* = argmin_x ||Ax - b(y)||^2
# ここで b(y) = y * [1, 2, 3, 4, 5]^T （yはスカラーパラメータ）
# Upper-level: min_y ||x*(y) - x_target||^2

# 真のターゲット
x_target = np.array([1.0, 2.0])

# 観測行列 A (5×2)
A = np.array([
    [1.0, 0.0],
    [0.5, 0.5],
    [0.0, 1.0],
    [0.3, 0.7],
    [0.8, 0.2],
])

def b_func(y):
    """パラメータyに依存する観測ベクトル"""
    return y * np.array([1.0, 2.0, 3.0, 4.0, 5.0])

def solve_lower_level(y):
    """Lower-level: x* = (A^T A)^{-1} A^T b(y)"""
    b = b_func(y)
    return np.linalg.solve(A.T @ A, A.T @ b)

def upper_level_cost(y):
    """Upper-level cost: ||x*(y) - x_target||^2"""
    x_star = solve_lower_level(y)
    return np.sum((x_star - x_target)**2)

# yを変えたときのコスト
y_range = np.linspace(0, 3, 100)
costs = [upper_level_cost(y) for y in y_range]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(y_range, costs, 'b-', linewidth=2)
ax.set_xlabel('Parameter y')
ax.set_ylabel('Upper-level cost $\\|x^*(y) - x_{target}\\|^2$')
ax.set_title('Bilevel Optimization: yを変えるとx*が変わり、コストが変化', fontweight='bold')
ax.grid(True, alpha=0.3)
y_opt_brute = y_range[np.argmin(costs)]
ax.axvline(y_opt_brute, color='red', linestyle='--', label=f'y* ≈ {y_opt_brute:.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## 4.2 Implicit Differentiation (陰関数微分)

**SLAM Handbook Section 4.2.3, Eq. 4.12**: 最適性条件 $\nabla_\mathbf{x}\mathcal{L}(\mathbf{x}^*,\mathbf{y}) = \mathbf{0}$ を$\mathbf{y}$で微分:

$$\frac{\partial^2 \mathcal{L}}{\partial \mathbf{x}^* \partial \mathbf{y}} + \frac{\partial^2 \mathcal{L}}{\partial \mathbf{x}^* \partial \mathbf{x}^*} \cdot \frac{\partial \mathbf{x}^*}{\partial \mathbf{y}} = \mathbf{0}$$

$$\Rightarrow \frac{\partial \mathbf{x}^*}{\partial \mathbf{y}} = -\left(\frac{\partial^2 \mathcal{L}}{\partial \mathbf{x}^* \partial \mathbf{x}^*}\right)^{-1} \frac{\partial^2 \mathcal{L}}{\partial \mathbf{x}^* \partial \mathbf{y}}$$

線形最小二乗 $\mathcal{L} = \|\mathbf{A}\mathbf{x} - \mathbf{b}(\mathbf{y})\|^2$ の場合:
- $\frac{\partial^2 \mathcal{L}}{\partial \mathbf{x} \partial \mathbf{x}} = 2\mathbf{A}^\top\mathbf{A}$ (Hessian)
- $\frac{\partial^2 \mathcal{L}}{\partial \mathbf{x} \partial \mathbf{y}} = -2\mathbf{A}^\top \frac{\partial \mathbf{b}}{\partial \mathbf{y}}$

In [ ]:
# =============================================================
# Implicit Differentiation の実装
# =============================================================

def implicit_diff_dx_dy(A, y):
    """Implicit differentiation: dx*/dy for ||Ax - b(y)||^2
    
    dx*/dy = -(A^T A)^{-1} A^T db/dy
    """
    db_dy = np.array([1.0, 2.0, 3.0, 4.0, 5.0])  # db(y)/dy
    H = A.T @ A  # Hessian (constant for linear LS)
    return np.linalg.solve(H, A.T @ db_dy)

def upper_level_gradient_implicit(y):
    """∂U/∂y via implicit differentiation (Eq. 4.3)
    
    ∇_y U = ∂U/∂y + (∂U/∂x*) · (∂x*/∂y)
    U = ||x*(y) - x_target||^2 → ∂U/∂x* = 2(x* - x_target)
    """
    x_star = solve_lower_level(y)
    dU_dx = 2 * (x_star - x_target)  # ∂U/∂x*
    dx_dy = implicit_diff_dx_dy(A, y)  # ∂x*/∂y (via IFT)
    return dU_dx @ dx_dy  # chain rule

# 数値微分で検証
def numerical_gradient(y, eps=1e-5):
    return (upper_level_cost(y + eps) - upper_level_cost(y - eps)) / (2 * eps)

y_test = 1.5
grad_implicit = upper_level_gradient_implicit(y_test)
grad_numerical = numerical_gradient(y_test)

print(f"=== y = {y_test} での勾配 ===")
print(f"Implicit differentiation: {grad_implicit:.6f}")
print(f"Numerical differentiation: {grad_numerical:.6f}")
print(f"差: {abs(grad_implicit - grad_numerical):.2e} ✓")

In [ ]:
# =============================================================
# Gradient descentでyを最適化
# =============================================================

y = 0.1  # 初期値（意図的にずらす）
lr = 0.05
history_y = [y]
history_cost = [upper_level_cost(y)]

for step in range(50):
    grad = upper_level_gradient_implicit(y)
    y = y - lr * grad
    history_y.append(y)
    history_cost.append(upper_level_cost(y))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_cost, 'b-o', markersize=3)
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Upper-level cost')
ax1.set_title('コストの収束', fontweight='bold')
ax1.set_yscale('log'); ax1.grid(True, alpha=0.3)

ax2.plot(y_range, costs, 'k-', alpha=0.3, linewidth=2)
ax2.plot(history_y, [upper_level_cost(yi) for yi in history_y], 'ro-', markersize=4,
         label='GD trajectory')
ax2.plot(history_y[-1], upper_level_cost(history_y[-1]), 'g*', markersize=15,
         label=f'y* = {history_y[-1]:.3f}')
ax2.set_xlabel('y'); ax2.set_ylabel('Cost')
ax2.set_title('Gradient Descent on Upper-Level', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

x_final = solve_lower_level(history_y[-1])
print(f"最適 y = {history_y[-1]:.4f}")
print(f"x*(y) = {x_final}")
print(f"x_target = {x_target}")
print(f"→ Implicit Differentiationにより最適化ソルバーを通して勾配を計算し、yを学習")

## 4.3 Unrolled Differentiation (展開微分)

**SLAM Handbook Section 4.2.1, Eq. 4.4-4.6**: 最適化の反復を関数合成として展開し、AutoDiffで微分。

$$\mathbf{x}_T = (\Phi_T \circ \cdots \circ \Phi_1 \circ \Phi_0)(\mathbf{y})$$

各 $\Phi_t$ はGD/GNの1ステップ。chain ruleで$\frac{\partial \mathbf{x}_T}{\partial \mathbf{y}}$を計算。

**Truncated版** (Eq. 4.8): メモリ節約のため最後のM反復のみ保持。

In [ ]:
# =============================================================
# Unrolled Differentiation の実装
# =============================================================

def unrolled_solve(y, T=10, eta=0.1):
    """GDをT回展開して解く + 各ステップのヤコビアンを記録"""
    b = b_func(y)
    x = np.zeros(2)  # 初期値
    
    # 順伝播（各ステップでのx_tとヤコビアンを保存）
    xs = [x.copy()]
    
    for t in range(T):
        grad_x = 2 * A.T @ (A @ x - b)  # ∂L/∂x
        x = x - eta * grad_x
        xs.append(x.copy())
    
    return x, xs

def unrolled_gradient(y, T=10, eta=0.1, eps=1e-5):
    """数値的にUnrolled gradientを計算（比較用）"""
    x_plus, _ = unrolled_solve(y + eps, T, eta)
    x_minus, _ = unrolled_solve(y - eps, T, eta)
    dx_dy = (x_plus - x_minus) / (2 * eps)
    x_star, _ = unrolled_solve(y, T, eta)
    dU_dx = 2 * (x_star - x_target)
    return dU_dx @ dx_dy

# 展開回数Tの影響を比較
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

y_test = 1.5
Ts = [1, 3, 5, 10, 20, 50]
grads_unrolled = []
grad_true = upper_level_gradient_implicit(y_test)

for T in Ts:
    g = unrolled_gradient(y_test, T=T)
    grads_unrolled.append(g)

ax1.bar(range(len(Ts)), grads_unrolled, color='steelblue', alpha=0.7)
ax1.axhline(grad_true, color='red', linestyle='--', linewidth=2, label=f'Implicit (exact): {grad_true:.4f}')
ax1.set_xticks(range(len(Ts)))
ax1.set_xticklabels([f'T={T}' for T in Ts])
ax1.set_ylabel('∂U/∂y')
ax1.set_title('Unrolled Gradient vs 展開回数T', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 解の収束
_, xs_full = unrolled_solve(y_test, T=50)
x_star_true = solve_lower_level(y_test)
errors = [np.linalg.norm(x - x_star_true) for x in xs_full]
ax2.semilogy(errors, 'b-o', markersize=3)
ax2.set_xlabel('GD Iteration')
ax2.set_ylabel('$\\|x_t - x^*\\|$')
ax2.set_title('Lower-level GDの収束', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("→ Unrolledは展開回数Tが少ないと近似的、T→∞でImplicitに一致")
print("→ Implicitは収束後の最適性条件を直接使うのでT不要だがHessianが必要")

## 4.4 マニフォールド上の微分 (Lie群ヤコビアン)

**SLAM Handbook Section 4.3, Eq. 4.23-4.25**: Lie群上の関数 $f(\boldsymbol{\chi})$ の微分。

### Right Jacobian

$$\frac{\partial f(\boldsymbol{\chi})}{\partial \boldsymbol{\chi}} = \mathbf{J}_R = \lim_{\boldsymbol{\tau}\to 0} \frac{f(\boldsymbol{\chi} \circ \text{Exp}(\boldsymbol{\tau})) \ominus f(\boldsymbol{\chi})}{\boldsymbol{\tau}}$$

摂動を **右から** 加えるバージョン（ローカル座標系）。

### Left Jacobian

$$\frac{\partial f(\boldsymbol{\chi})}{\partial \boldsymbol{\chi}} = \mathbf{J}_L = \lim_{\boldsymbol{\varepsilon}\to 0} \frac{f(\text{Exp}(\boldsymbol{\varepsilon}) \circ \boldsymbol{\chi}) \ominus f(\boldsymbol{\chi})}{\boldsymbol{\varepsilon}}$$

摂動を **左から** 加えるバージョン（グローバル座標系）。

In [ ]:
# =============================================================
# SO(3)上の数値ヤコビアンデモ
# =============================================================
from scipy.linalg import expm, logm

def so3_wedge(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def so3_vee(W):
    return np.array([W[2,1], W[0,2], W[1,0]])

def so3_exp(v):
    phi = np.linalg.norm(v)
    if phi < 1e-10:
        return np.eye(3) + so3_wedge(v)
    a = v / phi
    aw = so3_wedge(a)
    return np.eye(3) + np.sin(phi)*aw + (1-np.cos(phi))*(aw@aw)

def so3_log(R):
    cos_phi = np.clip((np.trace(R)-1)/2, -1, 1)
    phi = np.arccos(cos_phi)
    if phi < 1e-10:
        return so3_vee(R - np.eye(3))
    return phi / (2*np.sin(phi)) * so3_vee(R - R.T)

# -------------------------------------------------------
# f(R) = R^{-1} = R^T の Right Jacobian
# -------------------------------------------------------
# Right perturbation: f(R Exp(τ)) = (R Exp(τ))^T = Exp(τ)^T R^T = Exp(-τ) R^T
# delta = Log(f(R)^{-1} f(R Exp(τ))) = Log(R Exp(-τ) R^T)
#       = Log(Exp(-Ad(R)τ))           (∵ R Exp(v) R^T = Exp(Rv))
#       = -Ad(R) τ = -R τ
# ∴ J_R = -R (not -R^T!)

R = so3_exp([0.3, -0.2, 0.5])

# 数値 right Jacobian of f(R) = R^{-1}
eps = 1e-7
J_R_numerical = np.zeros((3, 3))
f_R = R.T  # f(R) = R^{-1}

for k in range(3):
    tau = np.zeros(3)
    tau[k] = eps
    R_perturbed = R @ so3_exp(tau)   # R ⊕ τ (right perturbation)
    f_perturbed = R_perturbed.T      # f(R⊕τ) = (R Exp(τ))^{-1}
    delta = so3_log(f_R.T @ f_perturbed)  # Log(f(R)^{-1} f(R⊕τ))
    J_R_numerical[:, k] = delta / eps

# 解析解の導出:
# f(R Exp(τ)) = Exp(-τ) R^T
# f(R)^{-1} f(R Exp(τ)) = R Exp(-τ) R^T = Exp(-Rτ)   (Adjoint property)
# Log(...) = -Rτ
# ∴ J_R = -R
J_R_analytic = -R

print("=== SO(3)上のRight Jacobian: f(R) = R^{-1} ===")
print(f"数値ヤコビアン:\n{J_R_numerical}")
print(f"\n解析ヤコビアン (-R):\n{J_R_analytic}")
print(f"\n差のノルム: {np.linalg.norm(J_R_numerical - J_R_analytic):.2e}")
assert np.linalg.norm(J_R_numerical - J_R_analytic) < 1e-5, "ヤコビアンが一致しません！"
print("✓ 一致確認")

## 4.5 演習問題

### 演習1: 非線形Lower-Level
Lower-levelを非線形（例: $r_i = \|x\|^2 - y_i$）に変えてImplicit Diffを実装してください。Hessianが定数でなくなる点に注意。

### 演習2: Truncated Unrolled
展開回数をT=50で全展開した場合とM=5の最後の部分だけ保持した場合で、upper-level勾配の精度を比較してください。

### 演習3: SO(3)上のGradient Descent
SLAM Handbook Example 4.4のVisual-Inertial回転推定を実装し、$R_{k+1} = R_k \text{Exp}(-\alpha \nabla f(R_k))$ でSO(3)上の勾配降下法を試してください。

## まとめ

| 手法 | 利点 | 欠点 |
|---|---|---|
| **Unrolled Diff** | 実装が簡単（AutoDiffだけ） | メモリ大、Tが少ないと不正確 |
| **Implicit Diff** | 正確、メモリ効率良い | Hessian必要（線形系を解く） |
| **Truncated Unrolled** | メモリ節約 | 近似的 |

- **Bilevel最適化**: NNのパラメータyを、最適化ソルバーの解x*を通してend-to-endで学習
- **Implicit Differentiation**: 最適性条件の陰関数定理で $\partial x^*/\partial y$ を計算
- **マニフォールド上の微分**: Right/Left Jacobianで Lie群上の関数を微分
- **実応用**: PyPose, Theseus 等のライブラリが実装を提供

### 次のNotebook
→ Ch05: Dense Map Representations (Occupancy Grid, SDF/TSDF)